In [1]:
# ============================================================
# STAGE 9 — FINAL PROJECT QA
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
TABLE_PATH = PROJECT_ROOT / "outputs" / "tables"
FIGURE_PATH = PROJECT_ROOT / "outputs" / "figures"

print("=" * 70)
print("TATA POWER EV CHARGING ANALYTICS — FINAL QA")
print("=" * 70)

# ------------------------------------------------------------
# 1. CHECK REQUIRED OUTPUT TABLES
# ------------------------------------------------------------

required_tables = [
    "intervention_customer_priority.csv",
    "intervention_segment_opportunity.csv",
    "intervention_city_opportunity.csv",
    "top_intervention_targets.csv",
    "experiment_assignments.csv",
    "experiment_group_results.csv",
    "experiment_segment_results.csv",
    "experiment_customer_outcomes.csv",
    "final_segment_recommendation.csv",
    "final_city_recommendation.csv",
    "final_business_recommendation.csv",
]

print("\n1. OUTPUT TABLE CHECK")
print("-" * 70)

missing_tables = []

for filename in required_tables:
    path = TABLE_PATH / filename

    if path.exists():
        df = pd.read_csv(path)
        print(f"✓ {filename:<45} {len(df):>8,} rows")
    else:
        print(f"✗ MISSING: {filename}")
        missing_tables.append(filename)


# ------------------------------------------------------------
# 2. CHECK FIGURES
# ------------------------------------------------------------

print("\n2. FIGURE CHECK")
print("-" * 70)

figures = list(FIGURE_PATH.glob("*.png"))

print(f"Figures found: {len(figures)}")

for f in sorted(figures):
    print(f"✓ {f.name}")


# ------------------------------------------------------------
# 3. EXPERIMENT CONSISTENCY CHECK
# ------------------------------------------------------------

print("\n3. EXPERIMENT CONSISTENCY")
print("-" * 70)

group_results = pd.read_csv(
    TABLE_PATH / "experiment_group_results.csv"
)

experiment = pd.read_csv(
    TABLE_PATH / "experiment_customer_outcomes.csv"
)

print("Experiment customers:", len(experiment))
print("Experiment groups:", experiment["experiment_group"].value_counts().to_dict())

treatment = group_results[
    group_results["experiment_group"] == "Treatment"
].iloc[0]

control = group_results[
    group_results["experiment_group"] == "Control"
].iloc[0]

print(f"Treatment customers: {int(treatment['customers']):,}")
print(f"Control customers:   {int(control['customers']):,}")

print(f"Treatment shift rate: {treatment['session_shift_rate']:.4%}")
print(f"Control shift rate:   {control['session_shift_rate']:.4%}")

observed_lift = (
    treatment["session_shift_rate"]
    - control["session_shift_rate"]
)

print(f"Observed lift:        {observed_lift:.4%}")


# ------------------------------------------------------------
# 4. ECONOMICS CONSISTENCY CHECK
# ------------------------------------------------------------

print("\n4. ECONOMICS CONSISTENCY")
print("-" * 70)

recommendation = pd.read_csv(
    TABLE_PATH / "final_business_recommendation.csv"
)

rec = dict(
    zip(
        recommendation["decision_area"],
        recommendation["value"]
    )
)

incremental_sessions = float(
    rec["Incremental shifted sessions"]
)

incremental_value = float(
    rec["Incremental value"]
)

intervention_cost = float(
    rec["Intervention cost"]
)

net_value = float(
    rec["Net incremental value"]
)

roi = float(
    rec["ROI"]
)

print(f"Incremental sessions: {incremental_sessions:,.0f}")
print(f"Incremental value:    ₹{incremental_value:,.2f}")
print(f"Intervention cost:    ₹{intervention_cost:,.2f}")
print(f"Net value:            ₹{net_value:,.2f}")
print(f"ROI:                  {roi:,.2f}%")

# Mathematical checks
net_check = incremental_value - intervention_cost
roi_check = (net_value / intervention_cost) * 100

print("\nMathematical validation:")

if abs(net_check - net_value) < 0.01:
    print("✓ Net incremental value reconciles")
else:
    print("✗ Net value mismatch")

if abs(roi_check - roi) < 0.1:
    print("✓ ROI reconciles")
else:
    print("✗ ROI mismatch")


# ------------------------------------------------------------
# 5. FINAL BUSINESS DECISION
# ------------------------------------------------------------

print("\n5. FINAL DECISION")
print("-" * 70)

print("Target segment :", rec["Target segment"])
print("Priority city  :", rec["Priority city"])
print("Decision       :", rec["Final scale decision"])


# ------------------------------------------------------------
# 6. FINAL QA STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)

if len(missing_tables) == 0:
    print("✓ ALL REQUIRED OUTPUT TABLES PRESENT")
else:
    print(f"✗ {len(missing_tables)} REQUIRED TABLE(S) MISSING")

if abs(net_check - net_value) < 0.01:
    print("✓ ECONOMICS RECONCILES")

if abs(roi_check - roi) < 0.1:
    print("✓ ROI RECONCILES")

print("\nFINAL PROJECT STATUS:")
print("STAGES 1–8 COMPLETE")
print("STAGE 9 QA RUN COMPLETE")

print("=" * 70)

TATA POWER EV CHARGING ANALYTICS — FINAL QA

1. OUTPUT TABLE CHECK
----------------------------------------------------------------------
✓ intervention_customer_priority.csv               8,000 rows
✓ intervention_segment_opportunity.csv                 4 rows
✓ intervention_city_opportunity.csv                   12 rows
✓ top_intervention_targets.csv                       100 rows
✓ experiment_assignments.csv                       4,661 rows
✓ experiment_group_results.csv                         2 rows
✓ experiment_segment_results.csv                       8 rows
✓ experiment_customer_outcomes.csv                 4,661 rows
✓ final_segment_recommendation.csv                     8 rows
✓ final_city_recommendation.csv                       12 rows
✓ final_business_recommendation.csv                   12 rows

2. FIGURE CHECK
----------------------------------------------------------------------
Figures found: 28
✓ charger_mix.png
✓ city_peak_demand.png
✓ control_vs_treatment_shift_rate